# 📓 Notebook 05 — Cloud, Container & Kubernetes

Container escape checks, image CVEs, IaC misconfig, secret scanning, K8s RBAC abuse, CI/CD secret theft.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 08:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB
overlay         236G   85G  152G  36% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED          
gpt-oss:20b         f38aa0c53da5    13 GB     47 minutes ago       
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    About an hour ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"11.4%","hits":362,"max_size":1000,"misses":2813,"size":21},"category_stats":{"additional":{"available":1,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":5,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":3,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":2813,"success_rate":"0.7%","system_metrics":{"cpu_percent":7.7,"disk_usage":35.9,"memory_percent":29.7,"network_io":{"bytes_recv":41652109712,"bytes_sent":169537557,"dropin":0,"dropout":0,"errin":0,

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
if ! command -v kubectl >/dev/null; then
  curl -fsSL -o /usr/local/bin/kubectl "https://dl.k8s.io/release/v1.30.0/bin/linux/amd64/kubectl" && chmod +x /usr/local/bin/kubectl || true
fi
pip -q install detect-secrets 2>/dev/null || true
if ! command -v trivy >/dev/null; then
  wget -q https://github.com/aquasecurity/trivy/releases/latest/download/trivy_0.58.0_Linux-64bit.tar.gz -O /tmp/t.tgz && tar -xzf /tmp/t.tgz -C /usr/local/bin trivy 2>/dev/null || true
fi
if ! command -v kube-bench >/dev/null; then
  wget -q https://github.com/aquasecurity/kube-bench/releases/latest/download/kube-bench_0.10.5_linux_amd64.tar.gz -O /tmp/kb.tgz && tar -xzf /tmp/kb.tgz -C /usr/local/bin kube-bench 2>/dev/null || true
fi
if ! command -v gitleaks >/dev/null; then
  wget -q https://github.com/gitleaks/gitleaks/releases/latest/download/gitleaks_8.21.2_linux_x64.tar.gz -O /tmp/gl.tgz && tar -xzf /tmp/gl.tgz -C /usr/local/bin gitleaks 2>/dev/null || true
fi
echo "cloud tools ready"


cloud tools ready


In [ ]:
%%bash
# --- local k8s (k3s, no systemd) ---
curl -sfL https://get.k3s.io | INSTALL_K3S_SKIP_START=true sh - 2>/dev/null || true
nohup k3s server --disable traefik >/var/log/k3s.log 2>&1 &
sleep 20
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
kubectl get nodes 2>&1 | head || echo "k3s starting in background"


[INFO]  Finding release for channel stable
[INFO]  Using v1.36.5+k3s1 as release
[INFO]  Downloading hash https://github.com/k3s-io/k3s/releases/download/v1.36.5%2Bk3s1/sha256sum-amd64.txt
[INFO]  Downloading binary https://github.com/k3s-io/k3s/releases/download/v1.36.5%2Bk3s1/k3s
[INFO]  Verifying binary download
[INFO]  Installing k3s to /usr/local/bin/k3s
[INFO]  Skipping installation of SELinux RPM
[INFO]  Skipping /usr/local/bin/kubectl symlink to k3s, already exists
[INFO]  Creating /usr/local/bin/crictl symlink to k3s
[INFO]  Skipping /usr/local/bin/ctr symlink to k3s, command exists in PATH at /usr/bin/ctr
[INFO]  Creating killall script /usr/local/bin/k3s-killall.sh
[INFO]  Creating uninstall script /usr/local/bin/k3s-uninstall.sh
[INFO]  env: Creating environment file /etc/systemd/system/k3s.service.env
[INFO]  systemd: Creating service file /etc/systemd/system/k3s.service
[INFO]  systemd: Enabling k3s unit
E1014 08:08:53.596036   84869 memcache.go:265] couldn't get current 

In [ ]:
%%bash
# --- targets ---
docker run -d --name wrongsecrets --network host jeroenwillemsen/wrongsecrets
docker run -d --name vulnerable-nginx --network host nginxdemos/hello
sleep 15
echo "wrongsecrets -> http://localhost:8080"
curl -s -o /dev/null -w "wrongsecrets:%{http_code}\n" http://localhost:8080

Container wrongsecrets started
Container vulnerable-nginx started
wrongsecrets -> http://localhost:8080
wrongsecrets:404


In [ ]:
# --- example attacks ---
!trivy fs --scanners vuln,secret,misconfig /content/hexstrike 2>/dev/null | head -30 || true
!kube-bench run --targets node 2>/dev/null | head -30 || true
!gitleaks detect -s /content -v 2>/dev/null | head -20 || true


In [ ]:
%%bash
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
curl -fsSL https://raw.githubusercontent.com/kubernetes-goat/kubernetes-goat/master/deploy-k8s.sh | bash 2>/dev/null | tail -20 || true
kubectl get pods -A 2>/dev/null | head -30 || echo "k8s ready"


curl: (22) The requested URL returned error: 404
